In [21]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import numpy as np

In [23]:
data = pd.read_excel("dataset/PCOS_data_without_infertility.xlsx")
data.head()

,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm),Unnamed: 44
0,11,11,0,20,71.0,163.0,NaN,15,80,20,...,0.0,0,110,80,7,15,17.0,20.0,6.0,NaN
1,28,28,0,20,68.0,152.0,NaN,17,72,20,...,0.0,0,110,80,3,2,10.0,11.0,10.0,NaN
2,40,40,0,20,74.0,171.0,NaN,13,74,16,...,0.0,0,110,70,6,12,13.0,12.0,0.0,NaN
3,240,240,0,20,56.0,159.0,NaN,13,72,18,...,0.0,1,120,80,10,5,15.0,18.0,8.0,NaN
4,298,298,0,20,52.0,150.0,NaN,15,72,18,...,1.0,0,110,70,7,9,13.0,17.0,9.6,NaN


In [24]:
data.drop(["Sl. No", "Patient File No.", "Blood Group", "BMI", "Pulse rate(bpm) ", "RR (breaths/min)",	"Hb(g/dl)", "Marraige Status (Yrs)",
           "Pregnant(Y/N)","No. of aborptions", "  I   beta-HCG(mIU/mL)","II    beta-HCG(mIU/mL)",	"FSH(mIU/mL)",	"LH(mIU/mL)",	"FSH/LH",
           "Hip(inch)"	,"Waist(inch)"	,"Waist:Hip Ratio"	,"TSH (mIU/L)",	"AMH(ng/mL)", "PRL(ng/mL)",	"Vit D3 (ng/mL)",
           "PRG(ng/mL)","RBS(mg/dl)", "BP _Systolic (mmHg)","BP _Diastolic (mmHg)",	"Follicle No. (L)",	"Follicle No. (R)"	,
           "Avg. F size (L) (mm)"	,"Avg. F size (R) (mm)"	,"Endometrium (mm)","Unnamed: 44"
], axis=1, inplace=True)
print(data.columns)
data.to_csv('new_data.csv', index=False)

Index(['PCOS (Y/N)', ' Age (yrs)', 'Weight (Kg)', 'Height(Cm) ', 'Cycle(R/I)',
       'Cycle length(days)', 'Weight gain(Y/N)', 'hair growth(Y/N)',
       'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)',
       'Fast food (Y/N)', 'Reg.Exercise(Y/N)'],
      dtype='object')


In [25]:
data = data.apply(pd.to_numeric, errors='coerce')
data['Cycle(R/I)'] = data['Cycle(R/I)'].replace({2: 0, 4: 1})
data = data.fillna(data.mean(numeric_only=True))
data["BMI"] = data["Weight (Kg)"] / ((data["Height(Cm) "]/100) ** 2)

In [26]:
import numpy as np
# Binary columns
binary_cols = [
    'Fast food (Y/N)',
    'Cycle(R/I)',
    'hair growth(Y/N)',
    'Skin darkening (Y/N)',
    'Hair loss(Y/N)',
    'Pimples(Y/N)',
    'Reg.Exercise(Y/N)'
]

# Fill binary columns with mode, not mean
for col in binary_cols:
    if data[col].mode().empty:
        data[col] = data[col].fillna(0)
    else:
        data[col] = data[col].fillna(data[col].mode()[0])

# Fill remaining numeric columns with median
for col in data.columns:
    if data[col].isnull().sum() > 0:
        data[col] = data[col].fillna(data[col].median())


# Final second pass fill
for col in data.columns:
    if data[col].isnull().sum() > 0:
        data[col] = data[col].fillna(data[col].median())

print("Total NaN left:", data.isnull().sum().sum())
print(data[['Cycle(R/I)', 'BMI']].head())

Total NaN left: 0
   Cycle(R/I)        BMI
0           0  26.722873
1           1  29.432133
2           1  25.306932
3           0  22.151023
4           0  23.111111


In [27]:
# Define Features (X) and Target (y)
# X = df.drop('PCOS (Y/N)', axis=1)
X = data[
    [
        ' Age (yrs)',
        'Weight (Kg)',
        'Height(Cm) ',
        'Fast food (Y/N)',
        'Cycle(R/I)',
        'hair growth(Y/N)',
        'Skin darkening (Y/N)',
        'Hair loss(Y/N)',
        'Pimples(Y/N)',
        'Reg.Exercise(Y/N)',
        'BMI'
    ]
]
y = data['PCOS (Y/N)']

print("Data Cleaning Complete. Features (X) and Target (y) are defined.")

Data Cleaning Complete. Features (X) and Target (y) are defined.


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data Split and Scaling Complete.")

# --- MODEL INITIALIZATION ---
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100,random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100,random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100,random_state=42)
}
print(f"Initialized {len(models)} classifiers.")

Data Split and Scaling Complete.
Initialized 6 classifiers.


In [52]:
# --- TRAINING AND EVALUATION ---
# !pip install tabulate
from sklearn.metrics import confusion_matrix
results = {'Model': [], 'Accuracy': [], 'AUC-ROC': [], 'F1 Score': []}

for name, model in models.items():
    if name in ["Logistic Regression", "K-Nearest Neighbors"]:
        # mask = ~pd.isnull(X_train_scaled).any(axis=1)
        # X_tr = X_train_scaled[mask]
        # y_tr = y_train[mask]
        X_te = X_test_scaled
    else:
        train_data = pd.concat([X_train, y_train], axis=1).dropna()
        X_tr = train_data.iloc[:, :-1]
        y_tr = train_data.iloc[:, -1]
        X_te = X_test
    
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    # print(f"\n{name} Confusion Matrix:\n{cm}")
    
    accuracy = accuracy_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)

    results['Model'].append(name)
    results['Accuracy'].append(accuracy)
    results['AUC-ROC'].append(auc_roc)
    results['F1 Score'].append(f1)

results_df = pd.DataFrame(results).sort_values(by='AUC-ROC', ascending=False).reset_index(drop=True)

print("--- Model Comparison Results ---\n")
print("Model | Accuracy | AUC-ROC | F1 Score")
print(results_df.to_markdown(index=False))

c:\Users\RESHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\RESHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\RESHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\RESHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


--- Model Comparison Results ---

Model | Accuracy | AUC-ROC | F1 Score
| Model               |   Accuracy |   AUC-ROC |   F1 Score |
|:--------------------|-----------:|----------:|-----------:|
| Logistic Regression |   0.834356 |  0.889365 |   0.769231 |
| AdaBoost            |   0.803681 |  0.867925 |   0.673469 |
| Random Forest       |   0.815951 |  0.857633 |   0.7      |
| Gradient Boosting   |   0.815951 |  0.854374 |   0.705882 |
| Decision Tree       |   0.748466 |  0.701201 |   0.594059 |
| K-Nearest Neighbors |   0.674847 |  0.5      |   0        |


In [53]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# model = LogisticRegression(max_iter = 1000)
# model.fit(X_train, y_train)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# model = LogisticRegression(max_iter=1000)
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("AUC-ROC:", roc_auc_score(y_test, y_prob))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

pickle.dump(model, open("models/pcos_model.pkl", "wb"))
pickle.dump(scaler, open("models/scaler.pkl", "wb"))

AUC-ROC: 0.8854202401372213
Accuracy: 0.8343558282208589
F1 Score: 0.7610619469026548


In [43]:
# import pickle
# pickle.dump(model, open("pcos_model.pkl", "wb"))
# pickle.dump(scaler, open("scaler.pkl", "wb"))

# print("✅ Model and Scaler Saved Successfully")

In [ ]:
# from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# y_pred = model.predict(X_test)
# y_prob = model.predict_proba(X_test)[:, 1]
# print("LogisticRegression")
# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("AUC-ROC:", roc_auc_score(y_test, y_prob))
# print("F1 Score:", f1_score(y_test, y_pred))

LogisticRegression
Accuracy: 0.8073394495412844
AUC-ROC: 0.8220946915351507
F1 Score: 0.7123287671232876


In [19]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

stack_model = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
        ("gb", GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, random_state=42)),
        ("lr", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))
    ],
    final_estimator=LogisticRegression(),
    cv=5,
    stack_method="predict_proba"
)

In [20]:
stack_model.fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]

from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

print("STACKING RESULTS")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print("F1:", f1_score(y_test, y_pred))

STACKING RESULTS
Accuracy: 0.8073394495412844
AUC: 0.8443328550932567
F1: 0.7123287671232876


In [40]:
def generate_recommendations(user_data, prediction):
    recommendations = {
        "diet": [],
        "exercise": [],
        "lifestyle": []
    }

    bmi = user_data["BMI"]
    fast_food = user_data["Fast_food"]
    cycle_length = user_data["Cycle_length"]
    weight_gain = user_data["Weight_gain"]
    
    # PCOS Prediction Based
    if prediction == 1:
        recommendations["lifestyle"].append(
            "Consult a gynecologist for further diagnosis"
        )
    # BMI Rules
    if bmi > 25:
        recommendations["diet"].append(
            "Follow a low glycemic index diet"
        )
        recommendations["exercise"].append(
            "30 minutes of cardio daily"
        )
    if bmi > 30:
        recommendations["exercise"].append(
            "Include strength training 3–4 times a week"
        )
    # Fast Food
    if fast_food == 1:
        recommendations["diet"].append(
            "Reduce processed and fast food intake"
        )
    # Irregular Cycle
    if cycle_length > 35:
        recommendations["lifestyle"].append(
            "Track menstrual cycle regularly"
        )
        recommendations["diet"].append(
            "Increase fiber-rich foods"
        )
    # Weight Gain
    if weight_gain == 1:
        recommendations["exercise"].append(
            "Include yoga and daily walking"
        )
    # General PCOS Advice
    recommendations["lifestyle"].append(
        "Maintain 7–8 hours of sleep"
    )

    recommendations["lifestyle"].append(
        "Practice stress management (meditation or yoga)"
    )
    return recommendations

In [41]:
user_data = {
    "Age": 24,
    "Weight": 70,
    "Height": 165,
    "BMI": 25.7,
    "Cycle_length": 40,
    "Fast_food": 1,
    "Weight_gain": 1,
    "Hair_growth": 0,
    "Skin_darkening": 0,
    "Pimples": 1,
    "Exercise": 0
}
input_df = pd.DataFrame([user_data])

# Prediction from ML model
prediction = model.predict(input_df)[0]

recommendations = generate_recommendations(user_data, prediction)

print(recommendations)

{'diet': ['Follow a low glycemic index diet', 'Reduce processed and fast food intake', 'Increase fiber-rich foods'], 'exercise': ['30 minutes of cardio daily', 'Include yoga and daily walking'], 'lifestyle': ['Consult a gynecologist for further diagnosis', 'Track menstrual cycle regularly', 'Maintain 7–8 hours of sleep', 'Practice stress management (meditation or yoga)']}


c:\Users\RESHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [42]:
pip install streamlit-option-menu

Note: you may need to restart the kernel to use updated packages.
